# LLM Investigation Reports (Evidence-Locked)

**Pipeline:** preprocessing → aml_graph_construction → train_baseline_comparison → pgexplainer_full → **this notebook** → final reports

- **Evidence-locked:** the prompt receives only the `explain_transaction()` output plus the transaction's own lookup row.
- **Hard identifier gate:** every transaction and account ID in the generated report is checked against the evidence after generation.
- **Greedy decoding** (`temperature=0`).
- **Metrics are self-tested** before any report is judged.
- **Model performance numbers** appear in the final summary, not in the prompt.

**Before running:** Ollama must be running (`ollama serve`) and the model must be pulled (`ollama pull llama3.1:8b`).

**Outputs** are saved to `DATA_DIR/llm_reports/`:
- `report_{TXN_ID}.md`: one investigation report per transaction
- `summary.json`: aggregate quality metrics
- `all_reports.json`: all report texts
- `pipeline_report.md`: the final pipeline summary

## 0. Imports & Config

In [1]:
import json, pickle, re, time, glob
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import PNAConv
from torch_geometric.utils import add_remaining_self_loops
from torch_geometric.explain.algorithm.utils import set_masks, clear_masks

VLLM_BASE_URL = "http://localhost:11434/v1"   # Ollama OpenAI-compatible endpoint
VLLM_MODEL    = "llama3.1:8b"                    # Ollama model tag
TEMPERATURE   = 0.0
MAX_TOKENS    = 800

N_REPORTS           = 10     
CANDIDATE_POOL_SIZE = 40
RANDOM_SEED         = 2000
IG_STEPS            = 50

# Auto-detect the multi_pna metrics.json. Set to a specific Path to override, or None to auto-detect.
MODEL_METRICS_PATH = None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/jovyan/conda-envs/aml-gnn/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. DATA_DIR resolution

In [2]:
CANDIDATE_DATA_DIRS = [
    Path("/home/jovyan/AML_Project/NoteBooks/Data/Bassam Data"),
    Path("/home/jovyan/AML_Project/Data/Bassam Data"),
    Path("/home/jovyan/AML_Project/NoteBooks/Bassam/Data/Bassam Data"),
]

REQUIRED_FILES = [
    "graph_data.pt", "metadata.json", "encoders.pkl", "edge_scaler.pkl",
    "transactions_lookup.csv", "transaction_ids.npy",
    "best_model_pna.pt", "best_hyperparams_pna.json",
    "pgexplainer_state.pt", "pgexplainer_config.json",
    "predicted_positive_edges.pt",
]

DATA_DIR = None
for cand in CANDIDATE_DATA_DIRS:
    if cand.exists() and all((cand / f).exists() for f in REQUIRED_FILES):
        DATA_DIR = cand
        break

if DATA_DIR is None:
    print("ERROR: Cannot find a directory with all required artifacts.")
    for cand in CANDIDATE_DATA_DIRS:
        if cand.exists():
            missing = [f for f in REQUIRED_FILES if not (cand / f).exists()]
            print(f"  {cand}: missing {missing}")
        else:
            print(f"  {cand}: does not exist")
    raise FileNotFoundError(
        "Run pgexplainer_full.py to completion first "
        "(it produces pgexplainer_state.pt, pgexplainer_config.json, "
        "predicted_positive_edges.pt)."
    )

OUT_DIR = DATA_DIR / "llm_reports"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"DATA_DIR: {DATA_DIR}")
print(f"DEVICE:   {DEVICE}")

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_DIR: /home/jovyan/AML_Project/NoteBooks/Data/Bassam Data
DEVICE:   cuda


## 2. Load artifacts (read-only)

In [3]:
data = torch.load(DATA_DIR / "graph_data.pt", weights_only=False)
with open(DATA_DIR / "metadata.json") as f:
    metadata = json.load(f)

edge_cat_start = data.edge_cat_start if hasattr(data, "edge_cat_start") \
    else metadata["edge_cat_start"]
EDGE_FEATURE_COLS = metadata["edge_feature_cols"]
CAT_COLS = ["Payment Format", "Payment Currency", "Receiving Currency"]
vocab_sizes = [metadata["vocab_sizes"][c] for c in CAT_COLS]
n_nodes = data.x.size(0)
in_dim = data.x.size(1)

with open(DATA_DIR / "encoders.pkl", "rb") as f:
    cat_encoders = pickle.load(f)
with open(DATA_DIR / "edge_scaler.pkl", "rb") as f:
    edge_scaler = pickle.load(f)

CONTINUOUS_EDGE_COLS = ["log_amount_paid", "log_amount_received", "log_amount_ratio",
                        "log_hours_since_last_tx", "log_tx_count_cumulative"]
SCALED_TO_RAW = {f"scaled_{c}": c for c in CONTINUOUS_EDGE_COLS}
RAW_TO_REAL = {c: c.replace("log_", "") for c in CONTINUOUS_EDGE_COLS}
cat_decoders = {col: {v: k for k, v in enc.items()}
                for col, enc in cat_encoders.items()}

transactions_lookup = pd.read_csv(DATA_DIR / "transactions_lookup.csv")
transaction_ids = np.load(DATA_DIR / "transaction_ids.npy", allow_pickle=True)
txn_id_to_edge_idx = {tid: i for i, tid in enumerate(transaction_ids)}
edge_idx_to_txn_id = {i: tid for tid, i in txn_id_to_edge_idx.items()}
txn_id_to_split = dict(zip(transactions_lookup["transaction_id"],
                           transactions_lookup["split"]))

with open(DATA_DIR / "pgexplainer_config.json") as f:
    _pg_cfg = json.load(f)
LOCKED_THRESHOLD = _pg_cfg.get("locked_threshold", 0.5)
FINAL_TOP_K = _pg_cfg.get("final_top_k", 5)

predicted_positive_test = torch.load(DATA_DIR / "predicted_positive_edges.pt",
                                     weights_only=False)["test"]

print(f"threshold={LOCKED_THRESHOLD:.4f}  top_k={FINAL_TOP_K}")
print(f"predicted-positive test pool: {predicted_positive_test.numel():,}")

threshold=0.9967  top_k=3
predicted-positive test pool: 3,000


## 3. Frozen PNA EdgeClassifier (verbatim)

In [4]:
class EdgeEncoder(nn.Module):
    def __init__(self, edge_cat_start, vocab_sizes, emb_dim=8, out_dim=32):
        super().__init__()
        self.edge_cat_start = edge_cat_start
        n_continuous = edge_cat_start + 1
        self.cont_proj = nn.Linear(n_continuous, out_dim)
        self.cat_embeddings = nn.ModuleList(
            [nn.Embedding(v, emb_dim) for v in vocab_sizes])
        self.out_proj = nn.Linear(out_dim + emb_dim * len(vocab_sizes), out_dim)

    def forward(self, edge_attr, is_reverse=None):
        if is_reverse is None:
            is_reverse = torch.zeros(edge_attr.size(0), 1,
                                     device=edge_attr.device,
                                     dtype=edge_attr.dtype)
        cont = torch.cat([edge_attr[:, :self.edge_cat_start], is_reverse], dim=1)
        cont_out = F.relu(self.cont_proj(cont))
        cat = edge_attr[:, self.edge_cat_start:].long()
        cat_out = torch.cat([emb(cat[:, i])
                             for i, emb in enumerate(self.cat_embeddings)], dim=1)
        return self.out_proj(torch.cat([cont_out, cat_out], dim=1))


class NodeEncoderPNA(nn.Module):
    def __init__(self, in_dim, hidden_dim, deg, n_layers=2, dropout=0.2,
                 edge_dim=32, towers=2):
        super().__init__()
        aggregators = ["mean", "min", "max", "std"]
        scalers = ["identity", "amplification", "attenuation"]
        conv_kwargs = dict(aggregators=aggregators, scalers=scalers, deg=deg,
                           edge_dim=edge_dim, towers=towers,
                           pre_layers=1, post_layers=1)
        self.convs = nn.ModuleList([PNAConv(in_dim, hidden_dim, **conv_kwargs)])
        for _ in range(n_layers - 1):
            self.convs.append(PNAConv(hidden_dim, hidden_dim, **conv_kwargs))
        self.dropout = dropout

    def forward(self, x, edge_index, edge_embedding):
        for i, conv in enumerate(self.convs):
            x = conv(x, edge_index, edge_embedding)
            if i < len(self.convs) - 1:
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
        return x


class EdgeClassifier(nn.Module):
    def __init__(self, model_name, in_dim, edge_cat_start, vocab_sizes,
                 hidden_dim=32, n_layers=2, dropout=0.2, deg=None, pna_towers=2):
        super().__init__()
        self.model_name = model_name
        self.hidden_dim = hidden_dim
        self.uses_edge_attr_in_mp = True
        self.edge_encoder = EdgeEncoder(edge_cat_start, vocab_sizes,
                                        emb_dim=8, out_dim=hidden_dim)
        assert deg is not None
        self.node_encoder = NodeEncoderPNA(in_dim, hidden_dim, deg,
                                           n_layers, dropout,
                                           edge_dim=hidden_dim, towers=pna_towers)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim * 3, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, 1),
        )

    def encode(self, x, edge_index, edge_attr, is_reverse=None):
        edge_emb = self.edge_encoder(edge_attr, is_reverse)
        return self.node_encoder(x, edge_index, edge_emb)

    def score(self, h_src, h_dst, target_edge_attr):
        target_edge_emb = self.edge_encoder(target_edge_attr)
        z = torch.cat([h_src, h_dst, target_edge_emb], dim=1)
        return self.head(z).squeeze(-1)


def augment_subgraph_traceable(edge_index, edge_attr, e_id, num_nodes):
    ei, ea, eid = edge_index, edge_attr, e_id.clone()
    is_reverse = torch.zeros(ei.size(1), 1, dtype=torch.float)
    non_self_loop = ei[0] != ei[1]
    rev_ei = ei[:, non_self_loop].flip(0)
    rev_ea = ea[non_self_loop]
    rev_eid = eid[non_self_loop]
    rev_is_reverse = torch.ones(rev_ei.size(1), 1, dtype=torch.float)
    ei = torch.cat([ei, rev_ei], dim=1)
    ea = torch.cat([ea, rev_ea], dim=0)
    eid = torch.cat([eid, rev_eid], dim=0)
    is_reverse = torch.cat([is_reverse, rev_is_reverse], dim=0)

    mask = ei[0] != ei[1]
    self_loop_mask = ~mask
    existing_loop_nodes = ei[0, self_loop_mask]
    existing_loop_ea = ea[self_loop_mask]
    existing_loop_eid = eid[self_loop_mask]
    existing_loop_is_rev = is_reverse[self_loop_mask]

    loop_index = torch.arange(num_nodes, dtype=ei.dtype, device=ei.device)
    loop_ei = loop_index.unsqueeze(0).repeat(2, 1)
    loop_ea = ea.new_full((num_nodes,) + ea.size()[1:], 0.0)
    loop_eid = torch.full((num_nodes,), -1, dtype=eid.dtype, device=eid.device)
    loop_is_reverse = torch.zeros(num_nodes, 1, dtype=torch.float)

    loop_ea[existing_loop_nodes] = existing_loop_ea
    loop_eid[existing_loop_nodes] = existing_loop_eid
    loop_is_reverse[existing_loop_nodes] = existing_loop_is_rev

    return (torch.cat([ei[:, mask], loop_ei], dim=1),
            torch.cat([ea[mask], loop_ea], dim=0),
            torch.cat([is_reverse[mask], loop_is_reverse], dim=0),
            torch.cat([eid[mask], loop_eid], dim=0))


with open(DATA_DIR / "best_hyperparams_pna.json") as f:
    best_hp = json.load(f)
NUM_NEIGHBORS_INFER = best_hp["num_neighbors"]

checkpoint = torch.load(DATA_DIR / "best_model_pna.pt", weights_only=False)
model = EdgeClassifier(
    "pna", in_dim, edge_cat_start, vocab_sizes,
    hidden_dim=checkpoint["hyperparameters"]["hidden_dim"],
    n_layers=checkpoint["hyperparameters"]["n_layers"],
    dropout=checkpoint["hyperparameters"]["dropout"],
    deg=checkpoint["pna_deg_histogram"],
).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()
for p in model.parameters():
    p.requires_grad_(False)
print(f"Frozen PNA loaded (seed {checkpoint['seed']}).")

Frozen PNA loaded (seed 1000).


## 4. Phase-scoped bases + `get_target_context`

In [5]:
val_base_mask = data.train_mask | data.val_mask
BASE_EDGES_BY_PHASE = {
    "train": (data.edge_index[:, data.train_mask], data.edge_attr[data.train_mask]),
    "val":   (data.edge_index[:, val_base_mask], data.edge_attr[val_base_mask]),
    "test":  (data.edge_index, data.edge_attr),
}


def _local_lookup(seed_ids, query_ids):
    sorted_ids, sort_perm = torch.sort(seed_ids)
    pos_in_sorted = torch.searchsorted(sorted_ids, query_ids)
    return sort_perm[pos_in_sorted]


def get_target_context(target_edge_idx, phase):
    base_ei, base_ea = BASE_EDGES_BY_PHASE[phase]
    src = data.edge_index[0, target_edge_idx].item()
    dst = data.edge_index[1, target_edge_idx].item()
    target_ea = data.edge_attr[target_edge_idx:target_edge_idx + 1].to(DEVICE)
    seed_nodes = torch.tensor([src, dst], dtype=torch.long)
    base_graph = Data(x=data.x, edge_index=base_ei, edge_attr=base_ea,
                      num_nodes=data.x.size(0))
    torch.manual_seed(target_edge_idx)
    np.random.seed(target_edge_idx % (2**32 - 1))
    loader = NeighborLoader(base_graph, num_neighbors=NUM_NEIGHBORS_INFER,
                             input_nodes=seed_nodes,
                             batch_size=seed_nodes.size(0), shuffle=False)
    sub = next(iter(loader))
    aug_ei, aug_ea, aug_is_rev, aug_eid = augment_subgraph_traceable(
        sub.edge_index, sub.edge_attr, sub.e_id, sub.num_nodes)
    x_dev = sub.x.to(DEVICE)
    aug_ei, aug_ea, aug_is_rev, aug_eid = (
        aug_ei.to(DEVICE), aug_ea.to(DEVICE),
        aug_is_rev.to(DEVICE), aug_eid.to(DEVICE))
    seed_ids = sub.n_id[:sub.batch_size]
    local_src = _local_lookup(seed_ids,
        torch.tensor([src], device=seed_ids.device)).to(DEVICE)
    local_dst = _local_lookup(seed_ids,
        torch.tensor([dst], device=seed_ids.device)).to(DEVICE)
    with torch.no_grad():
        h0 = model.encode(x_dev, aug_ei, aug_ea, aug_is_rev)
        p_original = torch.sigmoid(
            model.score(h0[local_src], h0[local_dst], target_ea)).item()
    return {"src": src, "dst": dst, "target_ea": target_ea, "x_dev": x_dev,
            "aug_ei": aug_ei, "aug_ea": aug_ea,
            "aug_is_rev": aug_is_rev, "aug_eid": aug_eid,
            "local_src": local_src, "local_dst": local_dst, "h0": h0,
            "p_original": p_original, "target_edge_idx": target_edge_idx}

## 5. Frozen PGExplainerMLP

In [6]:
class PGExplainerMLP(nn.Module):
    def __init__(self, hidden_dim, explainer_hidden):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim * 4, explainer_hidden), nn.ReLU(),
            nn.Linear(explainer_hidden, 1),
        )

    def forward(self, h_u, h_v, h_target_src, h_target_dst):
        z = torch.cat([h_u, h_v, h_target_src, h_target_dst], dim=1)
        return self.net(z).squeeze(-1)


_explainer_ckpt = torch.load(DATA_DIR / "pgexplainer_state.pt",
                             weights_only=False)
explainer_mlp = PGExplainerMLP(
    hidden_dim=_explainer_ckpt["hidden_dim"],
    explainer_hidden=_explainer_ckpt.get("explainer_hidden", 64)
).to(DEVICE)
explainer_mlp.load_state_dict(_explainer_ckpt["explainer_state_dict"])
explainer_mlp.eval()
for p in explainer_mlp.parameters():
    p.requires_grad_(False)
print("Frozen PGExplainer loaded.")

Frozen PGExplainer loaded.


## 6. Explanation machinery

In [7]:
def merge_and_filter(edge_index, e_id, importance_raw, edge_attr):
    is_synthetic_self_loop = (e_id == -1)
    keep = ~is_synthetic_self_loop
    kept_eid = e_id[keep].tolist()
    kept_scores = importance_raw[keep].tolist()
    merged = {}
    for eid, score in zip(kept_eid, kept_scores):
        if eid not in merged or score > merged[eid]:
            merged[eid] = score
    return merged


@torch.no_grad()
def get_explainer_scores(ctx):
    aug_ei, h0 = ctx["aug_ei"], ctx["h0"]
    h_src0, h_dst0 = h0[ctx["local_src"]], h0[ctx["local_dst"]]
    h_u, h_v = h0[aug_ei[0]], h0[aug_ei[1]]
    h_tgt_src_bcast = h_src0.expand(aug_ei.size(1), -1)
    h_tgt_dst_bcast = h_dst0.expand(aug_ei.size(1), -1)
    scores = torch.sigmoid(
        explainer_mlp(h_u, h_v, h_tgt_src_bcast, h_tgt_dst_bcast))
    merged = merge_and_filter(aug_ei, ctx["aug_eid"], scores, ctx["aug_ea"])
    merged.pop(ctx["target_edge_idx"], None)
    return merged


def _build_binary_mask(ctx, target_eids, mode):
    eid_list = ctx["aug_eid"].tolist()
    mask = torch.zeros(len(eid_list), device=DEVICE)
    for i, eid in enumerate(eid_list):
        if eid == -1:
            mask[i] = 1.0
        elif mode == "remove":
            mask[i] = 0.0 if eid in target_eids else 1.0
        elif mode == "keep_only":
            mask[i] = 1.0 if eid in target_eids else 0.0
    return mask


@torch.no_grad()
def _score_with_mask(ctx, mask):
    set_masks(model.node_encoder, mask, ctx["aug_ei"], apply_sigmoid=False)
    h = model.encode(ctx["x_dev"], ctx["aug_ei"],
                     ctx["aug_ea"], ctx["aug_is_rev"])
    p = torch.sigmoid(model.score(
        h[ctx["local_src"]], h[ctx["local_dst"]], ctx["target_ea"])).item()
    clear_masks(model.node_encoder)
    return p


def compute_fidelity_pair(ctx, topk_eids):
    p_without = _score_with_mask(ctx, _build_binary_mask(ctx, topk_eids, "remove"))
    p_only = _score_with_mask(ctx, _build_binary_mask(ctx, topk_eids, "keep_only"))
    return ctx["p_original"] - p_without, ctx["p_original"] - p_only


def integrated_gradients_target(ctx, steps=IG_STEPS):
    h_src0 = ctx["h0"][ctx["local_src"]].detach()
    h_dst0 = ctx["h0"][ctx["local_dst"]].detach()
    raw_ea = ctx["target_ea"].detach()
    cont_part = raw_ea[:, :edge_cat_start]
    cat_part = raw_ea[:, edge_cat_start:]
    baseline_cont = torch.zeros_like(cont_part)
    grads_accum = torch.zeros_like(cont_part)
    for step in range(1, steps + 1):
        alpha = step / steps
        interp_cont = (baseline_cont + alpha * (cont_part - baseline_cont)) \
            .clone().requires_grad_(True)
        interp_ea = torch.cat([interp_cont, cat_part], dim=1)
        edge_emb = model.edge_encoder(interp_ea)
        z = torch.cat([h_src0, h_dst0, edge_emb], dim=1)
        prob = torch.sigmoid(model.head(z).squeeze(-1))
        grads_accum = grads_accum + torch.autograd.grad(prob, interp_cont)[0]
    avg_grad = grads_accum / steps
    return ((cont_part - baseline_cont) * avg_grad).squeeze(0).detach().cpu()


def decode_transaction_features(edge_attr_row):
    row = edge_attr_row.cpu().numpy() if torch.is_tensor(edge_attr_row) \
        else np.asarray(edge_attr_row)
    decoded = {}
    scaled_cols = [c for c in EDGE_FEATURE_COLS[:edge_cat_start]
                   if c in SCALED_TO_RAW]
    if scaled_cols:
        scaled_vals = np.array(
            [[row[EDGE_FEATURE_COLS.index(c)] for c in scaled_cols]])
        raw_log_vals = edge_scaler.inverse_transform(scaled_vals)[0]
        raw_real_vals = np.expm1(raw_log_vals)
        for c, v in zip(scaled_cols, raw_real_vals):
            decoded[RAW_TO_REAL[SCALED_TO_RAW[c]]] = float(v)
    for c in EDGE_FEATURE_COLS[:edge_cat_start]:
        if c not in SCALED_TO_RAW:
            decoded[c] = float(row[EDGE_FEATURE_COLS.index(c)])
    for c in CAT_COLS:
        idx = EDGE_FEATURE_COLS.index(f"{c}_enc")
        decoded[c] = cat_decoders.get(c, {}).get(
            int(row[idx]), f"<UNK code {int(row[idx])}>")
    return decoded


def explain_transaction(transaction_id, top_k=None):
    if transaction_id not in txn_id_to_edge_idx:
        raise ValueError(f"Unknown transaction_id: {transaction_id}")
    edge_idx = txn_id_to_edge_idx[transaction_id]
    phase = txn_id_to_split.get(transaction_id)
    if phase not in BASE_EDGES_BY_PHASE:
        raise ValueError(f"Unexpected split for {transaction_id}: {phase!r}")
    k = top_k if top_k is not None else FINAL_TOP_K
    ctx = get_target_context(edge_idx, phase)
    merged_scores = get_explainer_scores(ctx)
    ranked = sorted(merged_scores.items(),
                    key=lambda kv: kv[1], reverse=True)[:k]
    contributing = [{"transaction_id": edge_idx_to_txn_id.get(eid),
                     "importance": round(float(s), 4),
                     "features": decode_transaction_features(data.edge_attr[eid])}
                    for eid, s in ranked]
    ig_attr = integrated_gradients_target(ctx)
    ig_cols = EDGE_FEATURE_COLS[:edge_cat_start]
    feature_attribution = {
        RAW_TO_REAL.get(SCALED_TO_RAW.get(c, c), SCALED_TO_RAW.get(c, c)):
        round(float(v), 4)
        for c, v in zip(ig_cols, ig_attr.tolist())}
    fid_plus, fid_minus = compute_fidelity_pair(
        ctx, set(eid for eid, _ in ranked))
    result = {
        "target_transaction": {
            "transaction_id": transaction_id,
            "features": decode_transaction_features(data.edge_attr[edge_idx]),
            "feature_attribution": feature_attribution,
        },
        "contributing_transactions": contributing,
        "model_prediction": {
            "probability": round(ctx["p_original"], 4),
            "predicted_label": int(ctx["p_original"] >= LOCKED_THRESHOLD),
        },
        "fidelity_check": {
            "fidelity_plus": round(fid_plus, 4),
            "fidelity_minus": round(fid_minus, 4),
        },
    }
    if ctx["p_original"] < LOCKED_THRESHOLD:
        result["confidence_warning"] = (
            f"probability {ctx['p_original']:.4f} is below the decision "
            f"threshold {LOCKED_THRESHOLD:.4f}")
    return result


print("explain_transaction() ready.")

explain_transaction() ready.


## 7. Select flagged test transactions

In [8]:
def select_transactions(n_reports, pool_size):
    assert predicted_positive_test.numel() >= n_reports, \
        f"Only {predicted_positive_test.numel()} predicted-positive — need {n_reports}"
    g = torch.Generator().manual_seed(RANDOM_SEED + 7)
    pool_idx = predicted_positive_test[
        torch.randperm(predicted_positive_test.size(0),
                       generator=g)[:pool_size]]
    pool_txn_ids = [edge_idx_to_txn_id[i.item()] for i in pool_idx]
    selected, n_skipped = [], 0
    for txn_id in pool_txn_ids:
        if len(selected) >= n_reports:
            break
        ctx = get_target_context(txn_id_to_edge_idx[txn_id],
                                 txn_id_to_split[txn_id])
        if ctx["p_original"] < LOCKED_THRESHOLD:
            n_skipped += 1
            continue
        selected.append(txn_id)
    assert len(selected) == n_reports, \
        f"Only confirmed {len(selected)}/{n_reports} — raise CANDIDATE_POOL_SIZE"
    print(f"Selected {len(selected)} transactions ({n_skipped} skipped).")
    return selected

## 8. Evidence builder 

In [9]:
def build_evidence(txn_id, result):
    row = transactions_lookup.loc[
        transactions_lookup["transaction_id"] == txn_id].iloc[0]
    features = result["target_transaction"]["features"]
    top_attr = sorted(result["target_transaction"]["feature_attribution"].items(),
                      key=lambda kv: abs(kv[1]), reverse=True)[:8]
    ev = {
        "transaction_id": txn_id,
        "model_prediction": {
            "probability": result["model_prediction"]["probability"],
            "predicted_label": "laundering"
                if result["model_prediction"]["predicted_label"] == 1
                else "normal",
            "decision_threshold": round(LOCKED_THRESHOLD, 4),
        },
        "target": {
            "raw_fields": {
                "Timestamp": str(row["Timestamp"]),
                "From Bank_Account": str(row["src_node"]),
                "To Bank_Account": str(row["dst_node"]),
                "Amount Paid": float(row["Amount Paid"]),
                "Amount Received": float(row["Amount Received"]),
                "Payment Currency": str(row["Payment Currency"]),
                "Receiving Currency": str(row["Receiving Currency"]),
                "Payment Format": str(row["Payment Format"]),
            },
            "feature_attribution": [
                {"feature": k, "raw_value": features.get(k), "importance": v}
                for k, v in top_attr if features.get(k) is not None
            ],
        },
        "neighborhood": [
            {"rank": r, "transaction_id": c["transaction_id"],
             "importance": c["importance"],
             "raw_fields": {k: c["features"][k]
                            for k in ("amount_paid", "Payment Format")
                            if k in c["features"]}}
            for r, c in enumerate(result["contributing_transactions"], 1)
        ],
    }
    if "confidence_warning" in result:
        ev["confidence_warning"] = True
    return ev

## 9. System prompt (SAR format + evidence-locking rules)

In [10]:
SYSTEM_PROMPT = """You are an experienced AML (Anti-Money-Laundering) compliance \
investigator. A graph-based fraud detection system flagged ONE specific \
transaction, and its explainability layer produced the EVIDENCE object you \
receive. Write a professional investigation report that goes BEYOND description.

## The evidence object
- "model_prediction": probability vs decision_threshold. If \
"confidence_warning" is true, the probability is close to or below the \
threshold — say so plainly.
- "target.raw_fields": the flagged transaction's own fields.
- "target.feature_attribution": which of its own features drove the decision. \
"importance" is SIGNED: positive pushed the flag, negative argued against it.
- "neighborhood": other transactions that most influenced the decision, \
ranked (rank 1 = most influential). It can be empty.

## Rules — no exceptions
1. Use ONLY facts that appear in the EVIDENCE. Never invent or infer a \
transaction ID, account, amount, date, or pattern name.
2. Cite transaction and account IDs exactly as they appear in the EVIDENCE.
3. Do NOT name laundering typologies (fan-out, fan-in, structuring, smurfing, \
layering, mule, cycle, ...) unless the word appears in the EVIDENCE. Describe \
the observed behavior in plain terms instead.
4. If the neighborhood is empty, or the evidence cannot explain something, \
say so instead of filling the gap.
5. Describe what the evidence shows; do not accuse anyone or state legal \
conclusions. Use "consider", "recommend", "suggest" language — never \
recommend freezing, blocking, or shutting down accounts.

## Analytical depth
Beyond stating facts: Interpret (what the evidence MEANS in this context), \
Assess (how serious and why), Connect (how the pieces fit together), \
Implicate (what could happen if this continues undetected). Every fact \
should be followed by its significance.

## Report format (SAR)
### Executive Summary
### Subject Information (WHO)
### Suspicious Activity (WHAT)
### Temporal Analysis (WHEN)
### Network Analysis (WHERE)
### Suspicion Basis (WHY)
### Recommended Actions

Start immediately — no preamble. Plain prose under each heading.
Keep the whole report under 450 words."""


def build_prompt(evidence):
    return SYSTEM_PROMPT, "EVIDENCE:\n" + json.dumps(
        evidence, indent=2, ensure_ascii=False)

In [11]:
%pip install -q openai

Note: you may need to restart the kernel to use updated packages.


In [13]:
import requests, json

with requests.post("http://localhost:11434/api/pull",
                   json={"model": "llama3.1:8b"}, stream=True) as r:
    for line in r.iter_lines():
        if line:
            d = json.loads(line)
            if "error" in d:
                print("ERROR:", d["error"]); break
            done, total = d.get("completed"), d.get("total")
            pct = f"{done/total:.0%}" if done and total else ""
            print(d.get("status"), pct)

pulling manifest 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c1932bc 
pulling 667b0c

## 10. LLM client (Ollama via OpenAI API, greedy, with retries)


In [14]:
from openai import OpenAI

client = OpenAI(base_url=VLLM_BASE_URL, api_key="EMPTY")
import requests
_base = VLLM_BASE_URL.rsplit("/v1", 1)[0]
try:
    _tags = requests.get(_base + "/api/tags", timeout=5)
except Exception as e:
    raise RuntimeError(f"Cannot reach the LLM server at {_base} ({e}). "
                       f"Start it with:  ollama serve")
try:
    _available = [m["name"] for m in _tags.json().get("models", [])]
except Exception:
    raise RuntimeError(f"Server at {_base} is not a normal Ollama server. "
                       f"/api/tags -> {_tags.status_code}: {_tags.text[:300]!r}")
if VLLM_MODEL not in _available:
    raise RuntimeError(
        f"Model '{VLLM_MODEL}' not found on the server. Available: {_available}. "
        f"Run:  ollama pull {VLLM_MODEL}   (or set VLLM_MODEL to one of the names above)")
print(f"[LLM] connected — {VLLM_MODEL}")


def llm_generate(system, user):
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=VLLM_MODEL,
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
            )
            text = resp.choices[0].message.content
            if text and text.strip():
                return text.strip()
        except Exception as e:
            print(f"  [LLM] attempt {attempt+1} failed: {e}")
            time.sleep(2)
    return None

[LLM] connected — llama3.1:8b


## 11. Evaluation metrics + self-tests

In [15]:
TXN_RE = re.compile(r"\bTXN\d{8}\b")
ACCT_RE = re.compile(r"\b\d+_[0-9A-Za-z]+\b")
DATE_RE = re.compile(r"\b\d{4}-\d{2}-\d{2}\b")
TIME_RE = re.compile(r"\b\d{1,2}:\d{2}(?::\d{2})?\b")
NUM_RE = re.compile(r"(?<![\w.])(\d[\d,]*(?:\.\d+)?)(\s?%)?")
TYPOLOGY_TERMS = ["fan-in", "fan-out", "fan in", "fan out",
                  "scatter-gather", "gather-scatter", "bipartite",
                  "smurfing", "structuring", "layering", "mule",
                  "shell company", "round-trip", "cycle"]
AGGRESSIVE_RE = re.compile(
    r"\b(freeze|freezing|frozen|immediately block|shut down)\b", re.I)
SECTIONS = {"summary": r"executive summary", "who": r"subject information",
            "what": r"suspicious activity", "when": r"temporal analysis",
            "where": r"network analysis", "why": r"suspicion basis",
            "actions": r"recommend"}
NEIGHBOR_ABSENT_CUES = ["no related", "no other", "no neighbor",
                        "no neighbourhood", "no neighborhood",
                        "not available", "isolated", "empty"]
LOW_CONF_CUES = ["below the", "under the", "close to", "near the",
                 "borderline", "caution", "low confidence",
                 "just below", "just under"]


def identifier_gate(evidence, text):
    ev_text = json.dumps(evidence, ensure_ascii=False)
    allowed_txn = set(TXN_RE.findall(ev_text))
    allowed_acct = set(ACCT_RE.findall(ev_text))
    cited_txn = set(TXN_RE.findall(text))
    cited_acct = set(ACCT_RE.findall(text))
    bad_txn = sorted(cited_txn - allowed_txn)
    bad_acct = sorted(cited_acct - allowed_acct)
    return {"hallucinated_txn": bad_txn,
            "hallucinated_accounts": bad_acct,
            "n_hallucinated": len(bad_txn) + len(bad_acct)}


def _close(a, e):
    return abs(a - e) <= max(5e-5, 5e-4 * abs(e))


def _numbers_in(t):
    vals = []
    for m in NUM_RE.finditer(t):
        try:
            vals.append(float(m.group(1).replace(",", "")))
        except ValueError:
            pass
    return vals


def figure_grounding(evidence, text):
    ev_text = json.dumps(evidence, ensure_ascii=False)
    allowed = _numbers_in(ev_text)
    t = ACCT_RE.sub(" ", TXN_RE.sub(" ", text))
    grounded, ungrounded = [], []
    for tok in DATE_RE.findall(t) + TIME_RE.findall(t):
        (grounded if tok in ev_text else ungrounded).append(tok)
    t = TIME_RE.sub(" ", DATE_RE.sub(" ", t))
    for m in NUM_RE.finditer(t):
        raw, pct = m.group(1), m.group(2)
        try:
            val = float(raw.replace(",", ""))
        except ValueError:
            continue
        if not (val >= 100 or "." in raw or pct):
            continue
        cands = [val] + ([val / 100.0] if pct else [])
        ok = any(_close(c, e) for c in cands for e in allowed)
        (grounded if ok else ungrounded).append(m.group(0))
    return {"grounded": grounded, "ungrounded": ungrounded}


def typology_claims(evidence, text):
    ev_l = json.dumps(evidence).lower()
    t_l = text.lower()
    return [w for w in TYPOLOGY_TERMS
            if re.search(rf"\b{re.escape(w)}s?\b", t_l) and w not in ev_l]


def rubric(evidence, text):
    tl = text.lower()
    words = len(text.split())
    checks = {name: bool(re.search(pat, tl))
              for name, pat in SECTIONS.items()}
    checks["reasonable_length"] = 120 <= words <= 500
    checks["not_a_refusal"] = not any(
        p in tl for p in ("i cannot", "i can't", "as an ai",
                          "i'm not able"))
    if evidence.get("neighborhood"):
        checks["cites_a_neighbor"] = any(
            str(n["transaction_id"]) in text
            for n in evidence["neighborhood"])
    else:
        checks["acknowledges_empty_neighborhood"] = any(
            c in tl for c in NEIGHBOR_ABSENT_CUES)
    if evidence.get("confidence_warning"):
        checks["acknowledges_low_confidence"] = any(
            c in tl for c in LOW_CONF_CUES)
    return {"checks": checks,
            "rubric_score": sum(checks.values()) / len(checks),
            "word_count": words,
            "has_aggressive_recommendations":
                bool(AGGRESSIVE_RE.search(text))}


def score_response(evidence, text):
    return {"identifier": identifier_gate(evidence, text),
            "figures": figure_grounding(evidence, text),
            "typology": typology_claims(evidence, text),
            "rubric": rubric(evidence, text)}


def run_self_tests():
    ev = {
        "transaction_id": "TXN00081234",
        "model_prediction": {"probability": 0.9983,
                             "predicted_label": "laundering",
                             "decision_threshold": 0.9967},
        "target": {
            "raw_fields": {
                "Timestamp": "2022-09-14 03:41:00",
                "From Bank_Account": "3402_8129331",
                "To Bank_Account": "3402_8129331",
                "Amount Paid": 48219.30,
                "Amount Received": 48219.30,
                "Payment Currency": "US Dollar",
                "Receiving Currency": "US Dollar",
                "Payment Format": "ACH"},
            "feature_attribution": [
                {"feature": "amount_paid", "raw_value": 48219.3,
                 "importance": 0.41},
                {"feature": "is_self_loop", "raw_value": 1.0,
                 "importance": 0.33}]},
        "neighborhood": [
            {"rank": 1, "transaction_id": "TXN00081201",
             "importance": 0.88,
             "raw_fields": {"amount_paid": 47800.10,
                            "Payment Format": "ACH"}}],
    }
    good = ("TXN00081234 was flagged with probability 0.9983 against a "
            "threshold of 0.9967. It is a transfer of 48,219.30 by ACH from "
            "3402_8129331 to itself, and the amount and the same-account "
            "behavior drove the decision. TXN00081201, a similar ACH "
            "transfer of 47,800.10, is the most influential related "
            "transaction.")
    s = score_response(ev, good)
    assert s["identifier"]["n_hallucinated"] == 0, s["identifier"]
    assert s["figures"]["ungrounded"] == [], s["figures"]
    assert s["typology"] == [], s["typology"]
    r = score_response(ev, good + " It also links to TXN00099999.")
    assert r["identifier"]["hallucinated_txn"] == ["TXN00099999"]
    r = score_response(ev, good + " Funds moved on to 3402_9999999.")
    assert r["identifier"]["hallucinated_accounts"] == ["3402_9999999"]
    assert "50,000" in score_response(
        ev, good.replace("48,219.30", "50,000"))["figures"]["ungrounded"]
    assert score_response(
        ev, good + " The pattern resembles fan-out structuring."
    )["typology"] == ["fan-out", "structuring"]
    assert score_response(
        ev, "We recommend freezing the account immediately."
    )["rubric"]["has_aggressive_recommendations"]
    print("  ✓ All metric self-tests passed.")


print("Running metric self-tests...")
run_self_tests()

Running metric self-tests...
  ✓ All metric self-tests passed.


## 12. Model performance loader

In [16]:
def find_metrics_json():
    if MODEL_METRICS_PATH:
        p = Path(MODEL_METRICS_PATH)
        return p if p.exists() else None
    for pattern in [
        "/home/jovyan/AML_Project/runs/*/metrics.json",
        "/home/jovyan/AML_Project/NoteBooks/Bassam/runs/*/metrics.json",
        "/home/jovyan/AML_Project/NoteBooks/runs/*/metrics.json",
    ]:
        matches = sorted(glob.glob(pattern))
        multi = [m for m in matches if "multi_pna" in m]
        if multi:
            return Path(multi[0])
    return None


def load_model_performance():
    p = find_metrics_json()
    if p is None:
        return None
    d = json.load(open(p))
    if "final_test_f1_mean" in d:
        return {"source": p.name, "model": d.get("winner", "pna"),
                "test_f1": round(d["final_test_f1_mean"], 4),
                "test_f1_std": d.get("final_test_f1_std"),
                "test_pr_auc": d.get("final_test_pr_auc_mean"),
                "protocol": "5-seed mean"}
    if "test_f1_at_thr" in d:
        return {"source": p.name, "model": d.get("model"),
                "test_f1": d["test_f1_at_thr"],
                "test_precision": d.get("test_precision_at_thr"),
                "test_recall": d.get("test_recall_at_thr"),
                "test_pr_auc": d.get("test_pr_auc"),
                "protocol": f"single run, seed {d.get('seed')}"}
    return None

## 13. Generate reports

In [17]:
def generate_reports(n_reports=N_REPORTS):
    selected_txn_ids = select_transactions(n_reports, CANDIDATE_POOL_SIZE)
    reports, results = [], []
    t_start = time.time()

    for i, txn_id in enumerate(selected_txn_ids):
        print(f"\n[{i+1}/{len(selected_txn_ids)}] {txn_id}")
        try:
            result = explain_transaction(txn_id)
        except Exception as e:
            print(f"  explain failed: {e}")
            continue

        evidence = build_evidence(txn_id, result)
        system, user = build_prompt(evidence)
        text = llm_generate(system, user)
        if text is None:
            print("  LLM generation failed — skipping")
            continue

        sc = score_response(evidence, text)
        y_true = int(transactions_lookup.loc[
            transactions_lookup["transaction_id"] == txn_id,
            "Is Laundering"].iloc[0])  # offline eval only — never in prompt

        is_valid = (sc["identifier"]["n_hallucinated"] == 0
                    and sc["rubric"]["checks"].get("summary")
                    and sc["rubric"]["checks"].get("why")
                    and sc["rubric"]["checks"].get("actions")
                    and not sc["rubric"]["has_aggressive_recommendations"])

        rec = {
            "transaction_id": txn_id,
            "y_true": y_true,
            "probability": evidence["model_prediction"]["probability"],
            "fidelity_plus": result["fidelity_check"]["fidelity_plus"],
            "fidelity_minus": result["fidelity_check"]["fidelity_minus"],
            "n_contributing": len(evidence["neighborhood"]),
            "identifier_gate_clean":
                sc["identifier"]["n_hallucinated"] == 0,
            "hallucinated_ids":
                sc["identifier"]["hallucinated_txn"]
                + sc["identifier"]["hallucinated_accounts"],
            "ungrounded_figures": sc["figures"]["ungrounded"],
            "typology_claims": sc["typology"],
            "has_aggressive_recommendations":
                sc["rubric"]["has_aggressive_recommendations"],
            "is_valid": bool(is_valid),
            "word_count": sc["rubric"]["word_count"],
            "rubric_score": round(sc["rubric"]["rubric_score"], 3),
        }
        results.append(rec)
        reports.append({"transaction_id": txn_id, "report": text})

        (OUT_DIR / f"report_{txn_id}.md").write_text(
            f"# Investigation Report — {txn_id}\n\n"
            f"**Model probability:** {rec['probability']}  |  "
            f"**Decision threshold:** {LOCKED_THRESHOLD:.4f}\n"
            f"**Identifier gate:** "
            f"{'PASS' if rec['identifier_gate_clean'] else 'FAIL — ' + str(rec['hallucinated_ids'])}\n\n"
            f"---\n\n{text}\n")
        print(f"  gate={'PASS' if rec['identifier_gate_clean'] else 'FAIL'}"
              f" | valid={rec['is_valid']}"
              f" | words={rec['word_count']}"
              f" | fid+={rec['fidelity_plus']:.4f}")

    return reports, results, t_start

## 14. Summary + final pipeline report

In [18]:
def write_outputs(reports, results, t_start):
    summary = {
        "model": VLLM_MODEL, "temperature": TEMPERATURE,
        "n_generated": len(reports),
        "n_valid": sum(r["is_valid"] for r in results),
        "n_gate_clean": sum(r["identifier_gate_clean"] for r in results),
        "n_aggressive": sum(r["has_aggressive_recommendations"]
                            for r in results),
        "avg_word_count": round(
            np.mean([r["word_count"] for r in results]), 0
        ) if results else 0,
        "elapsed_min": round((time.time() - t_start) / 60, 1),
        "results": results,
    }
    with open(OUT_DIR / "summary.json", "w") as f:
        json.dump(summary, f, indent=2, default=str)
    with open(OUT_DIR / "all_reports.json", "w") as f:
        json.dump(reports, f, indent=2, default=str)

    perf = load_model_performance()
    lines = [
        "# AML Pipeline — LLM Investigation Report Summary", "",
        f"Generated: {pd.Timestamp.now().isoformat()}", "",
        "## Models",
        f"- **Explained model (source of every report below):** "
        f"PNA EdgeClassifier — team pipeline "
        f"(`best_model_pna.pt`, seed {checkpoint['seed']}), "
        f"val-locked threshold {LOCKED_THRESHOLD:.4f}",
    ]
    if perf:
        if str(perf.get("model", "")).lower() != "pna":
            lines += [
                f"- **Comparison model (shared project):** "
                f"{perf['model']} — test F1@thr = {perf['test_f1']}"
                + (f", PR-AUC = {perf['test_pr_auc']}"
                   if perf.get("test_pr_auc") is not None else "")
                + f"  [{perf['protocol']}; source: {perf['source']}]",
                "  (performance context only — its predictions are NOT "
                "the ones explained here)",
            ]
        else:
            lines += [f"- Performance: test F1 = {perf['test_f1']}"]

    if results:
        lines += [
            "", "## Explanations (PGExplainer)",
            f"- {len(results)} flagged test transactions explained",
            f"- Mean fidelity+ = "
            f"{np.mean([r['fidelity_plus'] for r in results]):.4f}",
            f"- Mean fidelity- = "
            f"{np.mean([r['fidelity_minus'] for r in results]):.4f}",
            f"- Mean contributing transactions: "
            f"{np.mean([r['n_contributing'] for r in results]):.1f}",
            "", "## LLM report quality",
            f"- Valid (gate + structure + safety): "
            f"{summary['n_valid']}/{len(results)}",
            f"- Identifier gate clean: "
            f"{summary['n_gate_clean']}/{len(results)}",
            f"- Typology violations: "
            f"{sum(len(r['typology_claims']) for r in results)}",
            f"- Aggressive recommendations: "
            f"{summary['n_aggressive']} (target: 0)",
            f"- Average words: {summary['avg_word_count']:.0f}",
            "", "## Per-transaction", "",
            "| transaction | prob | fid+ | fid- | gate | valid |"
            " y_true (eval only) |",
            "|---|---|---|---|---|---|---|",
        ]
        for r in results:
            lines.append(
                f"| {r['transaction_id']} | {r['probability']} | "
                f"{r['fidelity_plus']:.3f} | {r['fidelity_minus']:.3f} | "
                f"{'PASS' if r['identifier_gate_clean'] else 'FAIL'} | "
                f"{r['is_valid']} | {r['y_true']} |")
    lines += [
        "", "## Integrity notes",
        "- Evidence-locked: every prompt contained only "
        "`explain_transaction()` output plus the transaction's own "
        "lookup row.",
        "- Ground-truth labels and pattern names were never given to "
        "the LLM; `y_true` above is used for offline evaluation only.",
        "- Reports generated with greedy decoding (temperature 0).",
        "",
        f"Elapsed: {summary['elapsed_min']} min",
    ]
    (OUT_DIR / "pipeline_report.md").write_text("\n".join(lines))

    print(f"\n{'='*60}")
    print(f"✅ Saved {len(reports)} reports to {OUT_DIR}/")
    print(f"   - report_{{TXN_ID}}.md  (per transaction)")
    print(f"   - summary.json         (aggregate metrics)")
    print(f"   - all_reports.json     (all texts)")
    print(f"   - pipeline_report.md   (final summary)")
    print(f"{'='*60}")
    return summary

## 15. Run

In [19]:
reports, results, t_start = generate_reports(N_REPORTS)
summary = write_outputs(reports, results, t_start)

Selected 10 transactions (1 skipped).

[1/10] TXN31893311
  gate=PASS | valid=True | words=294 | fid+=0.0007

[2/10] TXN31896026
  gate=PASS | valid=True | words=236 | fid+=0.0229

[3/10] TXN31891913
  gate=PASS | valid=True | words=315 | fid+=0.0231

[4/10] TXN31895031
  gate=PASS | valid=True | words=214 | fid+=0.3332

[5/10] TXN31894086
  gate=PASS | valid=True | words=298 | fid+=0.0049

[6/10] TXN31895568
  gate=PASS | valid=True | words=304 | fid+=0.0001

[7/10] TXN31895497
  gate=PASS | valid=True | words=256 | fid+=0.0030

[8/10] TXN31895151
  gate=PASS | valid=True | words=285 | fid+=0.0058

[9/10] TXN31892622
  gate=PASS | valid=True | words=291 | fid+=0.0055

[10/10] TXN31896978
  gate=PASS | valid=True | words=285 | fid+=0.0003

✅ Saved 10 reports to /home/jovyan/AML_Project/NoteBooks/Data/Bassam Data/llm_reports/
   - report_{TXN_ID}.md  (per transaction)
   - summary.json         (aggregate metrics)
   - all_reports.json     (all texts)
   - pipeline_report.md   (final sum

## 16. Quick look 

In [20]:
display(pd.DataFrame(results)[["transaction_id", "probability", "fidelity_plus",
        "fidelity_minus", "n_contributing", "identifier_gate_clean",
        "is_valid", "word_count", "rubric_score", "y_true"]])
print(reports[0]["report"] if reports else "no reports")

,transaction_id,probability,fidelity_plus,fidelity_minus,n_contributing,identifier_gate_clean,is_valid,word_count,rubric_score,y_true
0,TXN31893311,0.9997,0.0007,0.0035,3,True,True,294,1.0,1
1,TXN31896026,0.9970,0.0229,-0.0019,3,True,True,236,1.0,1
2,TXN31891913,0.9988,0.0231,-0.0002,3,True,True,315,1.0,1
3,TXN31895031,0.9997,0.3332,0.0002,3,True,True,214,1.0,1
4,TXN31894086,0.9999,0.0049,0.0005,3,True,True,298,1.0,1
5,TXN31895568,1.0000,0.0001,0.0011,3,True,True,304,1.0,1
6,TXN31895497,1.0000,0.0030,0.0009,3,True,True,256,1.0,1
7,TXN31895151,1.0000,0.0058,0.0006,3,True,True,285,1.0,1
8,TXN31892622,1.0000,0.0055,0.0010,3,True,True,291,1.0,1
9,TXN31896978,0.9992,0.0003,0.0058,3,True,True,285,1.0,1


**Executive Summary**

A high-confidence prediction of laundering activity has been made for transaction TXN31893311. The model's confidence level is 0.9997, which is above the decision threshold of 0.9967.

**Subject Information (WHO)**

The transaction in question involves accounts 001148_8013E8810 and 001216_801B4F730.

**Suspicious Activity (WHAT)**

The transaction is flagged due to its characteristics, which include:

* The amount paid and received are equal (12482.79 Euro).
* The transaction is in the same currency (Euro).
* The payment format is ACH.
* The transaction is between two different banks.

**Temporal Analysis (WHEN)**

The transaction occurred on September 18, 2022, at 01:36:00.

**Network Analysis (WHERE)**

The transaction is connected to three other transactions in the neighborhood, which are:

* TXN19965583: This transaction has a high importance score (1.0) and is ranked first in the neighborhood. It has a similar payment format (ACH) and a lower amount paid (48